In [ ]:
import os
import re
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy.stats import linregress
from pathlib import Path
from abc import ABCMeta, abstractmethod
from time import time
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression

In [ ]:
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.util import Logger, Util
from src.feature import *

In [ ]:
import importlib
import src.feature
importlib.reload(src.feature)
from src.feature import *

In [ ]:
pd.set_option("display.max_columns",500)
pd.set_option("display.max_rows", 500)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 処理実行

In [ ]:
def run_blocks(feature_blocks):
    print('start run blocks...')
    with Timer(prefix='run test'):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature()

In [ ]:
feature_blocks = [
    Key(use_cache=False, save_cache=True, logger=None),
	Target(use_cache=False, save_cache=True, logger=None),
    CategoryFeature(use_cache=False, save_cache=True, logger=None),
	CareerFeature(use_cache=False, save_cache=True, logger=None),
	UdemyActivityFeature(use_cache=False, save_cache=True, logger=None),
    UdemyTimeseriesFeature(use_cache=False, save_cache=True, logger=None),
    UdemyTitleEmbedding(use_cache=False, save_cache=True, logger=None),
	UdemyIDEmbedding(use_cache=False, save_cache=True, logger=None),
    UdemyCategorySimilarityFeature(use_cache=True, save_cache=True, logger=None),
    UdemyTitleSimilarityFeature(use_cache=True, save_cache=True, logger=None),
	DxFeature(use_cache=False, save_cache=True, logger=None),
    HrCategoryEmbeddingFeature(use_cache=False, save_cache=True, logger=None),
	HrNameEmbeddingFeature(use_cache=False, save_cache=True, logger=None),
    DxCategoryEmbeddingFeature(use_cache=False, save_cache=True, logger=None),
	DxNameEmbeddingFeature(use_cache=False, save_cache=True, logger=None),
	HrFeature(use_cache=False, save_cache=True, logger=None),
	OvertimeWorkByMonthFeature(use_cache=False, save_cache=True, logger=None),
    OvertimeWorkByMonthTimeseriesFeature(use_cache=True, save_cache=True, logger=None),
	PositionHistoryFeature(use_cache=False, save_cache=True, logger=None),
]

In [ ]:
run_blocks(feature_blocks)

In [ ]:
list_ = [
    'Key',
    'Target',
    'CategoryFeature',
    'CareerFeature',
    'UdemyActivityFeature',
    'UdemyTimeseriesFeature',
    'UdemyTitleEmbedding',
    'UdemyIDEmbedding',
    'UdemyCategorySimilarityFeature',
    'UdemyTitleSimilarityFeature',
    'DxFeature',
    'DxCategoryEmbeddingFeature',
    'DxNameEmbeddingFeature',
    'HrFeature',
    'HrCategoryEmbeddingFeature',
    'HrNameEmbeddingFeature',
    'OvertimeWorkByMonthFeature',
    'OvertimeWorkByMonthTimeseriesFeature',
    'PositionHistoryFeature',
]
dict_shape = {}
for feature_name in list_:
    dict_shape[feature_name] = Util.load_feature(feature_name).shape
pd.DataFrame(dict_shape, index=['n_rows', 'n_cols']).T

In [ ]:
df_dx = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_dx.pkl"))

In [ ]:
df_dx

In [ ]:
def create_sparse_matrix(df: pd.DataFrame, user_col: str, action_col: str, value_col=None,) -> tuple[sp.csr_matrix, LabelEncoder, LabelEncoder]:
    # user_col と action_col を数値に変更する
    user_encoder = LabelEncoder()
    action_encoder = LabelEncoder()
    user_array = user_encoder.fit_transform(df[user_col].to_numpy().ravel())
    action_array = action_encoder.fit_transform(df[action_col].to_numpy().ravel())

    # 重みを指定する (value_colがNoneの場合は1を指定)
    data_array = df[value_col].to_numpy().ravel() if value_col is not None else np.ones(len(df))

    # スパース行列を作成する
    sparse_matrix = sp.csr_matrix(
        (data_array, (user_array, action_array)),
        shape=(len(user_encoder.classes_), len(action_encoder.classes_)),
    )
    return sparse_matrix, user_encoder, action_encoder

def generate_embeddings(df: pd.DataFrame, user_col: str, action_col: str, value_col=None, n_components: int = 8) -> pd.DataFrame:
    """
    スパース行列を作成し、SVDで次元削減を行い、埋め込みを生成する関数
    Args:
        df (pd.DataFrame): 入力データフレーム
        user_col (str): ユーザーを識別するカラム名
        action_col (str): アクションを識別するカラム名
        value_col (str, optional): 重みを指定するカラム名 (デフォルトはNone)
        n_components (int): SVDでの次元数
    Returns:
        pd.DataFrame: ユーザーごとの埋め込み特徴量を含むデータフレーム
    """
    # スパース行列を作成
    sparse_matrix, user_encoder, action_encoder = create_sparse_matrix(df, user_col, action_col, value_col)

    # SVDで次元削減
    svd = TruncatedSVD(n_components=n_components, random_state=42)

    # ユーザー埋め込みを生成
    user_embeddings = svd.fit_transform(sparse_matrix)
    df_user_embeddings = pd.concat([
        pd.DataFrame({user_col: user_encoder.classes_}),
        pd.DataFrame(user_embeddings, columns=[f'svd_{action_col}_{i}' for i in range(user_embeddings.shape[1])])
    ], axis=1)

    # アクション埋め込みを生成
    action_embeddings = svd.components_.T
    course_title_to_vec = {
        course: action_embeddings[idx]
        for course, idx in zip(action_encoder.classes_, range(len(action_encoder.classes_)))
    }

    # 各ユーザーごとのベクトル平均を計算
    def compute_mean_embedding(group):
        embeddings = [course_title_to_vec[title] for title in group[action_col] if title in course_title_to_vec]
        if embeddings:
            return pd.Series(np.mean(embeddings, axis=0))
        else:
            return pd.Series([np.nan] * n_components)

    df_mean_embeddings = df.groupby(user_col).apply(compute_mean_embedding).reset_index()
    df_mean_embeddings.columns = [user_col] + [f"mean_svd_{action_col}_{i}" for i in range(n_components)]

    # 埋め込みデータをマージ
    df_embeddings = df[[user_col]].drop_duplicates().merge(df_user_embeddings, on=user_col, how='left')
    df_embeddings = df_embeddings.merge(df_mean_embeddings, on=user_col, how='left')

    return df_embeddings

In [ ]:
generate_embeddings(
    df=df_dx,
    user_col="社員番号",
    action_col="研修カテゴリ",
    value_col=None,
    n_components=10
)

In [ ]:
generate_embeddings(
    df=df_dx,
    user_col="社員番号",
    action_col="研修カテゴリ",
    value_col=None,
    n_components=10
)